# 04 – Feature Selection
Module owner: Lou-Ann Deley

Goal: find which features really matter for detecting phishing, and check if we can keep similar performance with fewer features.
We compare the three families seen in LAB 4:
- **Filter**: mutual information (`SelectKBest`);
- **Wrapper**: recursive feature elimination (`RFE`);
- **Embedded**: L1-penalised logistic regression (Lasso) and tree importances.

Rules: only `train.csv` is used, and every selection step is **inside a Pipeline**, so it is refitted on the training folds only (no data leakage).

In [ ]:
import time
from functools import partial
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.feature_selection import SelectKBest, mutual_info_classif, RFE, SelectFromModel
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.inspection import permutation_importance

SEED = 42
np.random.seed(SEED)
DATA_DIR = Path("../data")
FIG_DIR = Path("../figures"); FIG_DIR.mkdir(exist_ok=True)

train = pd.read_csv(DATA_DIR / "train.csv")
X_train, y_train = train.drop(columns="class"), train["class"]
FEATURES = list(X_train.columns)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

# Mutual information for discrete features (values -1/0/1)
mi_score = partial(mutual_info_classif, discrete_features=True, random_state=SEED)

# Final classifier: best model of notebook 03 (Gradient Boosting, tuned hyperparameters)
def make_clf():
    return HistGradientBoostingClassifier(learning_rate=0.1, max_iter=200, max_depth=6,
                                          l2_regularization=0.0, random_state=SEED)

def cv_scores(pipe):
    r = cross_validate(pipe, X_train, y_train, cv=cv, scoring=["roc_auc", "f1", "recall"])
    return {m: r["test_" + m].mean() for m in ["roc_auc", "f1", "recall"]}

ref = cv_scores(Pipeline([("clf", make_clf())]))
print("All 30 features:", {k: round(v, 3) for k, v in ref.items()})

## 1. Filter – mutual information
`SelectKBest` keeps the k features with the highest mutual information with the target. We try several values of k.

In [ ]:
K_LIST = [2, 5, 10, 15, 20, 30]
filter_res = {}
for k in K_LIST:
    pipe = Pipeline([("select", SelectKBest(mi_score, k=k)), ("clf", make_clf())])
    filter_res[k] = cv_scores(pipe)
filter_res = pd.DataFrame(filter_res).T.round(3)
filter_res

## 2. Wrapper – recursive feature elimination (RFE)
RFE trains a logistic regression, removes the least important feature, and repeats until k features are left. The kept features are then given to Gradient Boosting.

In [ ]:
t0 = time.time()
rfe_res = {}
for k in K_LIST:
    pipe = Pipeline([("select", RFE(LogisticRegression(max_iter=1000), n_features_to_select=k)),
                     ("clf", make_clf())])
    rfe_res[k] = cv_scores(pipe)
rfe_res = pd.DataFrame(rfe_res).T.round(3)
print(f"RFE time: {time.time() - t0:.1f}s")
rfe_res

In [ ]:
plt.plot(filter_res.index, filter_res["roc_auc"], "o-", label="Filter (mutual information)")
plt.plot(rfe_res.index, rfe_res["roc_auc"], "s-", label="Wrapper (RFE)")
plt.axhline(ref["roc_auc"], ls="--", color="gray", label="All 30 features")
plt.xlabel("Number of features kept (k)"); plt.ylabel("CV ROC-AUC (Gradient Boosting)")
plt.title("Performance vs number of features"); plt.legend()
plt.tight_layout(); plt.savefig(FIG_DIR / "fs_filter_vs_wrapper.png", dpi=150); plt.show()

## 3. Embedded – L1-penalised logistic regression (Lasso)
The L1 penalty sets some coefficients exactly to zero. The smaller `C`, the stronger the penalty, the fewer features are kept.
(`penalty="l1"` from the LAB 4 code is deprecated in recent scikit-learn; we use `l1_ratio=1` instead.)

In [ ]:
def l1_logreg(C):
    return LogisticRegression(l1_ratio=1, solver="liblinear", C=C, max_iter=1000)

l1_res = {}
for C in [0.01, 0.03, 0.1, 0.3, 1]:
    n_kept = SelectFromModel(l1_logreg(C)).fit(X_train, y_train).get_support().sum()
    pipe = Pipeline([("select", SelectFromModel(l1_logreg(C))), ("clf", make_clf())])
    l1_res[C] = {"n_features": n_kept, **cv_scores(pipe)}
l1_res = pd.DataFrame(l1_res).T
l1_res["n_features"] = l1_res["n_features"].astype(int)
l1_res.round(3)

## 4. Stability of the L1 selection (bootstrap)
We refit the L1 model on 50 bootstrap samples of the training set and count how often each feature is selected. A stable, useful feature is selected almost every time.

In [ ]:
rng = np.random.RandomState(SEED)
C_STAB = 0.03
counts = np.zeros(len(FEATURES))
for _ in range(50):
    idx = rng.choice(len(X_train), len(X_train), replace=True)
    sel = SelectFromModel(l1_logreg(C_STAB)).fit(X_train.iloc[idx], y_train.iloc[idx])
    counts += sel.get_support()
stability = pd.Series(counts / 50, index=FEATURES).sort_values()

ax = stability.plot.barh(figsize=(7, 9), color="tab:green")
ax.set_xlabel(f"Selection frequency over 50 bootstraps (L1, C = {C_STAB})")
ax.set_title("Stability of the L1 selection")
plt.tight_layout(); plt.savefig(FIG_DIR / "fs_l1_stability.png", dpi=150); plt.show()

## 5. Embedded – importance in the Gradient Boosting model
Permutation importance: we shuffle one feature and measure how much the ROC-AUC drops. It is computed on held-out folds, not on the data used for training.

In [ ]:
perm = np.zeros(len(FEATURES))
for tr, va in cv.split(X_train, y_train):
    model = make_clf().fit(X_train.iloc[tr], y_train.iloc[tr])
    r = permutation_importance(model, X_train.iloc[va], y_train.iloc[va], scoring="roc_auc",
                               n_repeats=5, random_state=SEED)
    perm += r.importances_mean / cv.get_n_splits()
perm = pd.Series(perm, index=FEATURES).sort_values()

ax = perm.plot.barh(figsize=(7, 9), color="tab:orange")
ax.set_xlabel("Drop in ROC-AUC when the feature is shuffled")
ax.set_title("Permutation importance (Gradient Boosting, held-out folds)")
plt.tight_layout(); plt.savefig(FIG_DIR / "fs_permutation_importance.png", dpi=150); plt.show()

## 6. Final ranking of the features
We combine the methods: mutual-information rank, L1 stability and permutation importance (each computed on the training set).

In [ ]:
mi_full = pd.Series(mi_score(X_train, y_train), index=FEATURES)
ranking = pd.DataFrame({
    "mutual_info": mi_full,
    "l1_stability": stability,
    "permutation_importance": perm,
})
ranking["mean_rank"] = ranking.rank(ascending=False).mean(axis=1)
ranking = ranking.sort_values("mean_rank")
ranking.round(3).head(15)

## 7. Data leakage check (LAB 4, Q5)
What happens if we select the features **before** cross-validation, on the whole training set? The validation folds were then used to choose the features, so the score is biased.

In [ ]:
k = 10
selector = SelectKBest(mi_score, k=k).fit(X_train, y_train)       # WRONG: selection on all data
leaky = cross_validate(make_clf(), X_train.loc[:, selector.get_support()], y_train,
                       cv=cv, scoring="roc_auc")["test_score"].mean()
correct = filter_res.loc[k, "roc_auc"]                              # selection inside the Pipeline
print(f"k = {k} | selection before CV (leaky): {leaky:.4f} | inside Pipeline (correct): {correct:.4f}")

Here the difference is small, because we have many more websites than features (n ≫ d). The bias becomes large when d ≫ n (e.g. genomics). We still keep the selection inside the Pipeline, because it is the only correct protocol.

## Conclusions
- With Gradient Boosting, 20 features (filter or RFE) give almost the same ROC-AUC as all 30 (≈ 0.990 vs 0.993). With 10 features the ROC-AUC drops to ≈ 0.984, and with 2 features to ≈ 0.95 or less: the last 10 features bring little, but the model still benefits from many weak signals.
- A small group of features comes first with every method: AnchorURL, HTTPS, PrefixSuffix-, then ServerFormHandler, SubDomains and WebsiteTraffic. They are the core signals of phishing in this dataset.
- With very few features, the filter is better than RFE (k = 2: 0.95 vs 0.89 ROC-AUC), because RFE relies on a linear model; from k = 10 they are equivalent. RFE is also slower.
- Selecting before cross-validation gives a slightly higher (biased) score; the effect is small here because n ≫ d.
- Feature selection does not improve performance here; its value is interpretability and a lighter model.
- All selection is done inside the Pipeline. Whether to keep all features or a reduced set will be decided in week 4, before the single test-set evaluation.